### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model 

## RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge


### Solution:

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

import numpy as np


### 1. Document Loading: Load documents from various sources

In [2]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """,
    """ 
    Artificial Intelligence 
    Artificial Intelligence (AI) is the simulation of human intelligence processes by machines,
    especially computer systems. These processes include learning (the acquisition of information
    and rules for using the information), reasoning (using rules to reach approximate or definite
    conclusions), and self-correction. AI applications include expert systems, natural language
    processing, speech recognition, and machine vision."""
    ,

    """
    Generative AI
    Generative AI refers to a class of artificial intelligence algorithms that can generate new
    content, such as text, images, music, or code, based on the data they have been trained on.
    Prominent examples of generative AI include Generative Adversarial Networks (GANs) and transformer-based models
    RAG (Retrieval-Augmented Generation) is a technique that combines retrieval-based methods with generative models to enhance
    the quality and relevance of generated content. It retrieves relevant information from a knowledge base
    to inform the generation process, improving accuracy and context-awareness.
    """
    ,
    """ 
    Socker Games
    Soccer, also known as football in many parts of the world, is a team sport played between two teams
    of eleven players with a spherical ball. It is the most popular sport globally, played by over 250 million players
    in over 200 countries. The objective of the game is to score by getting the ball into the opposing goal.
    """,
    """ 
    Climate Change
    Climate change refers to significant changes in global temperatures and weather patterns over time.
    While climate change is a natural phenomenon, scientific evidence shows that human activities,
    particularly the burning of fossil fuels, have been the primary drivers of recent global warming.   
    Effects of climate change include rising sea levels, more frequent extreme weather events,
    and disruptions to ecosystems and biodiversity.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [3]:
# Save sample documents to text files
import tempfile 

temp_dir = tempfile.mkdtemp()
# Note: tempfile.mkdtemp() creates a new unique directory, so we should not remove it right after creating it.
# If you need to reuse a specific path, remove it BEFORE creating the temp directory and then recreate it.

for i, doc in enumerate(sample_docs):
    file_path = os.path.join(temp_dir, f"doc_{i}.txt")
    with open(file_path, "w", encoding="utf8") as f:
        f.write(doc)
print(f"Sample documents saved to {temp_dir}/")
print(f"No of files in the temp directory: {len(os.listdir(temp_dir))}")

Sample documents saved to C:\Users\harry\AppData\Local\Temp\tmpvbhnsbus/
No of files in the temp directory: 7


In [4]:

# for i, doc in enumerate(sample_docs):
#     with open(f"data/doc_{i}.txt", "w") as f:
#         f.write(doc) 

In [5]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    temp_dir,  
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
# print(documents[0].page_content[:200] + "...")

documents 

Loaded 7 documents

First document preview:


[Document(metadata={'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of i

### 2. Document Splitting: Break documents into smaller chunks

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(chunks[0].page_content)


Created 10 chunks from 7 documents

Chunk example:
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through


### 3. Embedding Generation: Convert chunks into vector representations

In [7]:
from langchain_openai import OpenAIEmbeddings

sample_text = "Machine Learning is fascinating"
embeddings = OpenAIEmbeddings() 
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001C319961E80>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001C3242D0860>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [8]:
vector = embeddings.embed_query(sample_text)
print(f"Sample Text: {sample_text}")    
print(f"Embedding Vector (first 10 dimensions): {vector[:10]}")
vector

Sample Text: Machine Learning is fascinating
Embedding Vector (first 10 dimensions): [-0.02172444760799408, 0.016208980232477188, 0.010213345289230347, -0.022516079246997833, -0.0037213172763586044, 0.01783117651939392, 4.82096329506021e-05, 0.01027174387127161, -0.015547124668955803, -0.04134652763605118]


[-0.02172444760799408,
 0.016208980232477188,
 0.010213345289230347,
 -0.022516079246997833,
 -0.0037213172763586044,
 0.01783117651939392,
 4.82096329506021e-05,
 0.01027174387127161,
 -0.015547124668955803,
 -0.04134652763605118,
 0.007929293438792229,
 0.03628527745604515,
 -0.019128933548927307,
 -0.008234266191720963,
 -0.0013058676850050688,
 0.00581719446927309,
 0.03880292549729347,
 0.008811768144369125,
 -0.0005584409227594733,
 -0.008591149002313614,
 -0.031224025413393974,
 0.022048886865377426,
 -0.005914526060223579,
 -0.03441650792956352,
 -0.014898247085511684,
 0.0023018959909677505,
 0.003834871109575033,
 -0.03885483369231224,
 -0.012523352168500423,
 -0.002739888848736882,
 0.027590306475758553,
 -0.004736811853945255,
 -0.0170655008405447,
 -0.03981517627835274,
 -0.008513283915817738,
 -0.012211889959871769,
 -0.004152821376919746,
 0.0028583090752363205,
 -0.01670212857425213,
 -0.00013068816042505205,
 0.020076295360922813,
 0.02541007660329342,
 -0.008435418829

In [9]:
chunks

[Document(metadata={'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a sub

### 4. Vector Storage: Store embeddings in ChromaDB

Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [10]:
# Create a Chromadb vector store and add the chunks
from langchain.vectorstores import Chroma


# Create a Chroma vector store in the parent folder (one level up)
persist_directory = os.path.abspath(os.path.join(os.getcwd(), "..", "VectorDBs","ChromaDB"))
os.makedirs(persist_directory, exist_ok=True) 

# Initialize (or reinitialize) the Chroma vector store using the existing Chroma and embeddings objects
vector_store = Chroma.from_documents(   
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="kcj-langchain-rag-chromaDB"
)

print(f"Vector store created with {vector_store._collection.count()} vectors.")
print(f"Persisted to '{persist_directory}' directory.")

Vector store created with 10 vectors.
Persisted to 'd:\krishna-codejournal\kcj-langchain-rag-starter\src\LangChain\VectorDBs\ChromaDB' directory.


### Similarity Search: Find relevant chunks from vector store
### Test Similarity Search

In [11]:
query = "Explain the different types of machine learning."
retrieved_docs = vector_store.similarity_search(query, k=3) 
print(f"\nTop {len(retrieved_docs)} documents retrieved for the query: '{query}'")
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Retrieved Document {i+1} ---")
    print(f"Content: {doc.page_content[:500]}...")
    print(f"Metadata: {doc.metadata}")


Top 3 documents retrieved for the query: 'Explain the different types of machine learning.'

--- Retrieved Document 1 ---
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through...
Metadata: {'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_0.txt'}

--- Retrieved Document 2 ---
Content: Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized

In [12]:
query = "What is NLP"
retrieved_docs = vector_store.similarity_search(query, k=2)
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Retrieved Document {i+1} ---")
    print(f"Content: {doc.page_content[:500]}...")
    print(f"Metadata: {doc.metadata}")


--- Retrieved Document 1 ---
Content: Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text....
Metadata: {'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_2.txt'}

--- Retrieved Document 2 ---
Content: Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neura

In [13]:
query = "Describe deep learning and its applications."
retrieved_docs = vector_store.similarity_search(query, k=2) 
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Retrieved Document {i+1} ---")
    print(f"Content: {doc.page_content[:500]}...")
    print(f"Metadata: {doc.metadata}")


--- Retrieved Document 1 ---
Content: Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers...
Metadata: {'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_1.txt'}

--- Retrieved Document 2 ---
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learnin

In [14]:
query = "What are the effects of climate change?"
retrieved_docs = vector_store.similarity_search(query, k=2)
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Retrieved Document {i+1} ---")
    print(f"Content: {doc.page_content[:500]}...")
    print(f"Metadata: {doc.metadata}")


--- Retrieved Document 1 ---
Content: Climate Change
    Climate change refers to significant changes in global temperatures and weather patterns over time.
    While climate change is a natural phenomenon, scientific evidence shows that human activities,
    particularly the burning of fossil fuels, have been the primary drivers of recent global warming.   
    Effects of climate change include rising sea levels, more frequent extreme weather events,
    and disruptions to ecosystems and biodiversity....
Metadata: {'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_6.txt'}

--- Retrieved Document 2 ---
Content: Artificial Intelligence 
    Artificial Intelligence (AI) is the simulation of human intelligence processes by machines,
    especially computer systems. These processes include learning (the acquisition of information
    and rules for using the information), reasoning (using rules to reach approximate or definite
    conclusions), and self-correction. AI app

In [15]:
query = "What are the effects of air pollution?"
retrieved_docs = vector_store.similarity_search(query, k=2)
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Retrieved Document {i+1} ---")
    print(f"Content: {doc.page_content[:500]}...")
    print(f"Metadata: {doc.metadata}")


--- Retrieved Document 1 ---
Content: Climate Change
    Climate change refers to significant changes in global temperatures and weather patterns over time.
    While climate change is a natural phenomenon, scientific evidence shows that human activities,
    particularly the burning of fossil fuels, have been the primary drivers of recent global warming.   
    Effects of climate change include rising sea levels, more frequent extreme weather events,
    and disruptions to ecosystems and biodiversity....
Metadata: {'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_6.txt'}

--- Retrieved Document 2 ---
Content: Artificial Intelligence 
    Artificial Intelligence (AI) is the simulation of human intelligence processes by machines,
    especially computer systems. These processes include learning (the acquisition of information
    and rules for using the information), reasoning (using rules to reach approximate or definite
    conclusions), and self-correction. AI app

### Advanced Similarity Search With Scores

In [16]:
result_scores = vector_store.similarity_search_with_score("What are the effects of air pollution?", k=2)
result_scores

for doc, score in result_scores:
    print(f"\nScore: {score}")
    print(f"Content: {doc.page_content[:500]}...")
    print(f"Metadata: {doc.metadata}")


Score: 0.4070765972137451
Content: Climate Change
    Climate change refers to significant changes in global temperatures and weather patterns over time.
    While climate change is a natural phenomenon, scientific evidence shows that human activities,
    particularly the burning of fossil fuels, have been the primary drivers of recent global warming.   
    Effects of climate change include rising sea levels, more frequent extreme weather events,
    and disruptions to ecosystems and biodiversity....
Metadata: {'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpvbhnsbus\\doc_6.txt'}

Score: 0.5563536286354065
Content: Artificial Intelligence 
    Artificial Intelligence (AI) is the simulation of human intelligence processes by machines,
    especially computer systems. These processes include learning (the acquisition of information
    and rules for using the information), reasoning (using rules to reach approximate or definite
    conclusions), and self-correction. AI applicati

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [17]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    temperature=0,
    model_name="gpt-3.5-turbo"
)

In [18]:
response = llm.invoke("What is RAG?")
response

AIMessage(content='RAG stands for Red, Amber, Green. It is a color-coded system used to indicate the status or progress of a project, task, or situation. \n\n- Red typically indicates that there are significant issues or problems that need immediate attention.\n- Amber signifies that there are some concerns or risks that need to be addressed.\n- Green indicates that everything is on track and progressing as planned.\n\nRAG status is commonly used in project management, risk assessment, and performance reporting to quickly communicate the status of various activities or initiatives.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 12, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None,

In [19]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model("openai:gpt-3.5-turbo", temperature=0)
llm.invoke("What is RAG?")

AIMessage(content='RAG stands for Red, Amber, Green. It is a color-coded system used to indicate the status or progress of a project, task, or situation. \n\n- Red typically indicates that there are significant issues or problems that need immediate attention.\n- Amber signifies that there are some concerns or risks that need to be addressed.\n- Green indicates that everything is on track and progressing as planned.\n\nThe RAG system is commonly used in project management, risk assessment, and performance monitoring to quickly communicate the status of various activities.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 12, 'total_tokens': 116, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'ch

In [20]:
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

In [21]:
retriver = vector_store.as_retriever(search_kwargs={"k": 2})
retriver

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C32580A510>, search_kwargs={'k': 2})

In [22]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [23]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

##### What is create_stuff_documents_chain?
create_stuff_documents_chain creates a chain that "stuffs" (inserts) all retrieved documents into a single prompt and sends it to the LLM. It's called "stuff" because it literally stuffs all the documents into the context window at once.

In [24]:
documents_chain = create_stuff_documents_chain(
    llm=llm,    
    prompt=prompt
)

In [25]:
documents_chain 

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001C3296E87D0>, async_client=<openai.res

This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

#### What is create_retrieval_chain?
create_retrieval_chain is a function that combines a retriever (which fetches relevant documents) with a document chain (which processes those documents with an LLM) to create a complete RAG pipeline.

In [26]:
from langchain.chains import create_retrieval_chain

rag_chain = create_retrieval_chain(retriver, documents_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C32580A510>, search_kwargs={'k': 2}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf yo

In [27]:
response = rag_chain.invoke({"input": "What is RAG?"})
response["answer"]

'RAG stands for Retrieval-Augmented Generation. It is a technique that combines retrieval-based methods with generative models to enhance the quality and relevance of generated data. RAG is used to improve the performance of generative AI algorithms by incorporating retrieval-based approaches.'

In [28]:
response = rag_chain.invoke({"input": "What is AI?"})
response["answer"]

'AI, or Artificial Intelligence, is the simulation of human intelligence processes by machines, particularly computer systems. It involves tasks like learning, reasoning, and self-correction to perform various functions. AI applications include expert systems, natural language processing, speech recognition, and machine vision.'

In [29]:
response = rag_chain.invoke({"input": "What is Database Management System?"})
response["answer"]

'A Database Management System (DBMS) is a software that allows users to create, maintain, and manipulate databases. It provides an interface for users to interact with the database by entering data, querying information, and generating reports. DBMS also ensures data integrity, security, and efficient retrieval of data stored in the database.'

In [30]:
response = rag_chain.invoke({"input": "What is Blockchain?"})
response["answer"]

'Blockchain is a decentralized, distributed ledger technology that securely records transactions across multiple computers. Each block in the chain contains a list of transactions, and once added, it cannot be altered without changing all subsequent blocks. Blockchain is commonly associated with cryptocurrencies like Bitcoin but has applications beyond digital currencies, such as supply chain management and voting systems.'

In [31]:
# Function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Using create_retrieval_chain approach
    result = rag_chain.invoke({"input": question})
    
    print(f"Answer: {result['answer']}")
    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")
    
    return result

# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "="*80 + "\n")

Question: What are the three types of machine learning?
--------------------------------------------------
Answer: The three main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data for training, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through a system of rewards and punishments.

Retrieved Context:

--- Source 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...

--- Source 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...


Question: What is deep learning and how does it relate to neural networks?
----------------------------------------